## Changing CNN Filter Size

This example constructs a classifier for the CIFAR-10 dataset using a number of different filter sizes.

In [ ]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from timer import Timer

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and Prepare the Data

In [ ]:
# Define the transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the dataset into memory
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Function to load data into GPU
def load_data_to_gpu(dataset):
    data_loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    data_iter = iter(data_loader)
    images, labels = next(data_iter)
    return images.to('cuda'), labels.to('cuda')

# Load the train and test sets to GPU
train_images, train_labels = load_data_to_gpu(trainset)
test_images, test_labels = load_data_to_gpu(testset)

# Create TensorDatasets with data on GPU
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)

# Create DataLoaders with smaller batch sizes for training and testing
trainloader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)


### Build CNN Model

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, filter_size=3):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=filter_size, padding='same')
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=filter_size, padding='same')
        self.conv3 = nn.Conv2d(64, 64, kernel_size=filter_size, padding='same')
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 64) 
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
no_epochs = 200

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = batch_y
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = batch_y
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
loss_fn = nn.CrossEntropyLoss()

### Loop over filter sizes

In [ ]:
history_dict = dict()
timer_dict = dict()
timer = Timer()


for ifilter in range(3, 8, 2):
    cnn_model = CNNModel(ifilter).to(device)
    optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
    print("Training Filter: {0} X {1}".format(ifilter, ifilter))
    timer.start()
    history_dict["{0} X {1}".format(ifilter, ifilter)] = train_model(cnn_model, trainloader, testloader,
                                                                        loss_fn, optimizer, no_epochs)
    timer_dict["{0} X {1}".format(ifilter, ifilter)] = timer.stop()

### Factorized Filter

Here we use another network with factorized Covolutions - that is a 1 x n Conv layer followed by a n x 1 Conv layer.

In [ ]:
class CNNFactorizedModel(nn.Module):
    def __init__(self, filter_size=3):
        super(CNNFactorizedModel, self).__init__()
        self.conv1_1 = nn.Conv2d(3, 32, kernel_size=(1, filter_size), padding='same')
        self.conv1_2 = nn.Conv2d(32, 32, kernel_size=(filter_size, 1), padding='same')
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2_1 = nn.Conv2d(32, 64, kernel_size=(1, filter_size), padding='same')
        self.conv2_2 = nn.Conv2d(64, 64, kernel_size=(filter_size, 1), padding='same')
        self.conv3_1 = nn.Conv2d(64, 64, kernel_size=(1, filter_size), padding='same')
        self.conv3_2 = nn.Conv2d(64, 64, kernel_size=(filter_size, 1), padding='same')
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 64)  # Adjusted to account for input size
        self.fc2 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1_1(x))
        x = self.relu(self.conv1_2(x))
        x = self.pool(x)
        x = self.relu(self.conv2_1(x))
        x = self.relu(self.conv2_2(x))
        x = self.pool(x)
        x = self.relu(self.conv3_1(x))
        x = self.relu(self.conv3_2(x))
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
cnn_model = CNNFactorizedModel().to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

timer.start()
history_dict["1 x 3, 3 x 1"] = train_model(cnn_model, trainloader, testloader,
                                           loss_fn, optimizer, no_epochs)
timer_dict["1 x 3, 3 x 1"] = timer.stop()

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
def smoothed_points(points, factor=0.8):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points

In [ ]:
fig, ax = plt.subplots(figsize=(14,10))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    val_loss_hist = history_dict[ilr]
    val_loss_values = val_loss_hist['test_acc']
    ax.plot(epochs, smoothed_points(val_loss_values), label = ilr, markevery=10)

ax.set_xlabel('Epochs')
ax.set_ylabel('Validation Accuracy')
ax.legend()
ax.set_ylim(60, 75)
fig.savefig('TestFilterSizes.png', dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd

In [ ]:
timing_table = pd.DataFrame.from_dict(timer_dict, orient='index', columns=[['Training Time (s)']])

In [ ]:
timing_table

In [ ]:
timing_table.to_latex()